# CS116 S6E10 — Overnight 2× GPU members (clean OOF)

**Hai experiments độc lập**, chạy tuần tự và sử dụng chung `data/folds.csv` của nhóm:

- `quan_xgb_te_v1`: **XGBoost GPU + 21 Target Encoding** (pilot Fold 0 AUC **0.960005**).
- `quan_xgb_context_v1`: **XGBoost GPU + TE + conditional TE + frequency + exact-route profile** (thí nghiệm nghiên cứu, **chưa có điểm xác nhận**).

### Chuẩn bị trước khi nhấn Save Version
1. Vào Kaggle Notebook mới **Import Notebook** file này (cũng có thể ghép mã vào notebook hiện tại).
2. **Add Input**: competition `playground-series-s6e10`; dataset `quannh206/fold-data`; output notebook `quannh206/cs116` (phải có `cs116-kaggle-s6e10/src/data.py`). **Không tải dataset bằng tay.**
3. Bật **Accelerator: GPU T4 x2** hoặc ít nhất GPU T4. Code sử dụng **GPU 0**; không cần cả hai GPU.
4. Vào **Save Version → Save & Run All**, để máy Kaggle chạy. Khi hoàn thành xem **Versions → Output Files → `S6E10_Overnight_members.zip`**.

**Không xóa notebook hoặc dữ liệu cũ.** Nếu `S6E10_GPU_members.zip` cũ có trong input, notebook sẽ đưa sáu `.npy` cũ vào ZIP mới; nếu không có, vẫn xuất hai member mới bình thường.

**An toàn CV:** luôn dùng team `load_data()` và frozen folds; `TargetEncoder.fit_transform` cross-fitting trên **outer training only**; 10% early stopping nằm **bên trong** outer train; **không** sử dụng outer validation để chọn số cây; test prediction = trung bình 5 fold models. Hai thí nghiệm không blend/stack/submit.

> Lưu ý: Save & Run All chạy lại toàn bộ notebook trong phiên riêng. Kaggle có giới hạn thời gian chạy; file checkpoint theo fold hữu ích trong phiên chạy, nhưng chỉ được giữ lâu dài nếu phiên bản Kaggle lưu thành công. Không có cam kết thời gian chạy hoặc mức tăng AUC.

In [1]:
from pathlib import Path
import sys, os, gc, csv, time, json, shutil, zipfile, subprocess, warnings
from datetime import datetime, timezone
import numpy as np
import pandas as pd

# ===== THÍ NGHIỆM QUA ĐÊM =====
RUN_XGB_TE = True
RUN_XGB_CONTEXT = True
# Nếu muốn chỉ chạy pilot đã tốt: đặt RUN_XGB_CONTEXT = False

MAX_TREES = 3000
EARLY_STOP_ROUNDS = 120
WORK_PROJECT = Path('/kaggle/working/cs116-kaggle-s6e10')
WORK_SRC = WORK_PROJECT/'src'
DATA_DIR = WORK_PROJECT/'data'
OUT_DIR = WORK_SRC/'oof'
PROGRESS_DIR = WORK_PROJECT/'progress_overnight'
OUT_ZIP = Path('/kaggle/working/S6E10_Overnight_members.zip')
MANIFEST = Path('/kaggle/working/S6E10_Overnight_manifest.json')
for p in (WORK_PROJECT, WORK_SRC, DATA_DIR, OUT_DIR, PROGRESS_DIR):
    p.mkdir(parents=True, exist_ok=True)

try:
    import xgboost as xgb
    from xgboost import XGBClassifier
    from sklearn.preprocessing import TargetEncoder
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import roc_auc_score
except ImportError as e:
    raise RuntimeError('Required packages missing. Use Kaggle default image with recent xgboost/sklearn.') from e

print('XGBoost', xgb.__version__)
import sklearn
print('scikit-learn', sklearn.__version__)
assert hasattr(sklearn.preprocessing, 'TargetEncoder'), 'Need sklearn >= 1.3'
print('GPU check:')
subprocess.run(['nvidia-smi', '--query-gpu=index,name,memory.total', '--format=csv,noheader'],check=True)
# Fail fast: GPU is a requirement, not an invisible CPU fallback.
import torch
assert torch.cuda.is_available(), 'Bật GPU T4 trong Kaggle Settings trước khi Save & Run All'
print('GPU available:',torch.cuda.get_device_name(0))

XGBoost 3.4.1
scikit-learn 1.6.1
GPU check:
0, Tesla T4, 15360 MiB
1, Tesla T4, 15360 MiB
GPU available: Tesla T4


In [2]:
# ===== DÙNG ĐÚNG TEAM load_data() / folds.csv =====
PROJECT_INPUT = Path('/kaggle/input/notebooks/quannh206/cs116/cs116-kaggle-s6e10')
ORIGINALS = {
    'train.csv': Path('/kaggle/input/competitions/playground-series-s6e10/train.csv'),
    'test.csv': Path('/kaggle/input/competitions/playground-series-s6e10/test.csv'),
    'folds.csv': Path('/kaggle/input/datasets/quannh206/fold-data/folds.csv'),
}
for name, path in ORIGINALS.items():
    assert path.is_file(), f'Add Input: {path}'
    target = DATA_DIR/name
    if target.is_symlink(): target.unlink()
    elif target.exists():
        # Keep only a correct previous link; never silently read stale actual data.
        raise FileExistsError(f'Unexpected real file at {target}; use a clean Kaggle session')
    target.symlink_to(path)
assert (PROJECT_INPUT/'src'/'data.py').is_file(), f'Add Input: old CS116 notebook output ({PROJECT_INPUT})'
shutil.copytree(PROJECT_INPUT/'src', WORK_SRC, dirs_exist_ok=True,
                ignore=shutil.ignore_patterns('__pycache__','*.pyc','oof'))
for mod in ('data','utils','config'): sys.modules.pop(mod,None)
sys.path.insert(0,str(WORK_SRC))
import data as team_data
team_data.DATA_DIR = DATA_DIR
train,test = team_data.load_data()

fold_df = pd.read_csv(ORIGINALS['folds.csv'],usecols=['id','fold'])
assert train.shape==(699635,24),train.shape
assert test.shape==(299844,22),test.shape
assert np.array_equal(train.id.to_numpy(),fold_df.id.to_numpy())
assert np.array_equal(train.fold.to_numpy(),fold_df.fold.to_numpy())
assert set(train.fold.unique())==set(range(5))
assert train.id.is_unique and test.id.is_unique
Y=train['satisfaction'].astype(np.int32).to_numpy()
assert set(np.unique(Y))=={0,1}
FOLDS=train.fold.to_numpy(dtype=np.int8)
FEATURES=[c for c in test.columns if c!='id']
CATS=['Gender','Customer Type','Type of Travel','Class']
assert len(FEATURES)==21 and all(c in FEATURES for c in CATS)
NUMS=[c for c in FEATURES if c not in CATS]
print('Loaded via team data.py:',train.shape,test.shape)
print('Fold counts:',pd.Series(FOLDS).value_counts().sort_index().to_dict())

Loaded via team data.py: (699635, 24) (299844, 22)
Fold counts: {0: 139927, 1: 139927, 2: 139927, 3: 139927, 4: 139927}


In [3]:
# ===== FEATURES KHÔNG SỬ DỤNG NHÃN =====
start_features = time.perf_counter()
raw=pd.concat([train[FEATURES],test[FEATURES]],ignore_index=True)
n_train=len(train)

# Chỉ feature data, KHÔNG có satisfaction; được phép thống kê trên train+test.
base=pd.get_dummies(raw,columns=CATS,dummy_na=True,dtype=np.float32)
BASE_ARRAY=base.to_numpy(dtype=np.float32)
BASE_DIM=BASE_ARRAY.shape[1]
del base

# Target Encoding 21 features (numerical values are treated as exact categorical tokens).
TE_BASE=pd.DataFrame(index=raw.index)
for col in FEATURES:
    TE_BASE[col]=raw[col].astype('string').fillna('__MISSING__').astype(str)

# Optional context-enhanced member: 1 traveler segment, 10 conditional ratings,
# 5 conditional age/distance features = 16 additional TE columns.
TE_CONTEXT=TE_BASE.copy()
seg=TE_BASE['Type of Travel']+'|'+TE_BASE['Customer Type']+'|'+TE_BASE['Class']
TE_CONTEXT['segment_TCC']=seg
SCORES=[
    'Inflight wifi service','Online boarding','Inflight entertainment',
    'Seat comfort','Ease of Online booking','Cleanliness',
    'On-board service','Leg room service','Checkin service','Baggage handling'
]
for col in SCORES:
    TE_CONTEXT['ctx_'+col]=seg+'|'+TE_BASE[col]
for col,group in [
    ('Flight Distance','Class'),('Flight Distance','Type of Travel'),
    ('Flight Distance','Customer Type'),('Age','Class'),('Age','Type of Travel')]:
    TE_CONTEXT['ctx_'+col+'|'+group]=TE_BASE[col]+'|'+TE_BASE[group]

# Frequency features from unlabeled train+test ONLY (no target leakage).
count_source=['Flight Distance','Age','Class','Type of Travel','Customer Type',
              'Online boarding','Inflight wifi service','segment_TCC']
count_frame=TE_CONTEXT[count_source]
freq_columns={}
for col in count_source:
    vc=count_frame[col].value_counts(dropna=False)
    freq_columns['freq_'+col]=np.log1p(
        1e6 * count_frame[col].map(vc).to_numpy(dtype=np.float32) / len(raw)
    ).astype(np.float32)

# Route profile: exact Flight Distance aggregate ratings, only X (not y).
# Use leave-one-out group statistics: own rating is excluded from its route mean.
r=raw['Flight Distance']
route_cnt=r.groupby(r,dropna=False).transform('size').to_numpy(dtype=np.float32)
freq_columns['route_log_count']=np.log1p(route_cnt).astype(np.float32)
for col in ['Online boarding','Inflight wifi service','Seat comfort',
            'Inflight entertainment','Cleanliness','Ease of Online booking']:
    rating=raw[col].astype(np.float32)
    grp_sum=rating.groupby(r,dropna=False).transform('sum').to_numpy(dtype=np.float32)
    vals=rating.to_numpy(dtype=np.float32)
    glob=float(rating.mean())
    route_mean=np.where(route_cnt>1, (grp_sum-vals)/np.maximum(route_cnt-1,1), glob).astype(np.float32)
    freq_columns['route_mean_'+col]=route_mean
    freq_columns['route_resid_'+col]=(vals-route_mean).astype(np.float32)
EXTRA_ARRAY=pd.DataFrame(freq_columns).to_numpy(dtype=np.float32)

del raw, count_frame, count_source, freq_columns, seg, r, route_cnt, rating, grp_sum, vals, route_mean

def get_base(idx): return BASE_ARRAY[idx]
def get_extra(idx): return EXTRA_ARRAY[idx]

print('base dim:',BASE_DIM,'TE base:',TE_BASE.shape[1],
      'TE context:',TE_CONTEXT.shape[1],'non-target context:',EXTRA_ARRAY.shape[1])
print('Feature preparation minutes:',round((time.perf_counter()-start_features)/60,2))

base dim: 30 TE base: 21 TE context: 37 non-target context: 21
Feature preparation minutes: 0.2


In [4]:
# ===== TRAINING + CHECKPOINT + OUTPUT =====

def encoder(seed):
    return TargetEncoder(target_type='binary',smooth='auto',
                         cv=5,shuffle=True,random_state=seed)

def learner(n_estimators,seed,early_stop):
    return XGBClassifier(
        n_estimators=n_estimators,
        learning_rate=0.035,
        max_depth=6,
        min_child_weight=10,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=5.0,
        objective='binary:logistic',
        eval_metric='auc',
        tree_method='hist',
        device='cuda:0',
        random_state=seed,
        n_jobs=4,
        early_stopping_rounds=EARLY_STOP_ROUNDS if early_stop else None,
    )

RUN_META={
 'project':'CS116 Kaggle S6E10',
 'shared_folds':'quannh206/fold-data/folds.csv',
 'started_utc':datetime.now(timezone.utc).isoformat(),
 'models':{}
}

KNOWN_OLD_ZIPS=[
 Path('/kaggle/input/notebooks/quannh206/cs116/S6E10_GPU_members.zip'),
 Path('/kaggle/input/notebooks/quannh206/cs116/cs116-kaggle-s6e10/S6E10_GPU_members.zip'),
]

def package_results():
    # Important: incomplete progress is NOT published under member OOF names.
    MANIFEST.write_text(json.dumps(RUN_META,indent=2,ensure_ascii=False),encoding='utf-8')
    with zipfile.ZipFile(OUT_ZIP,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=1) as dst:
        names=set()
        for old in KNOWN_OLD_ZIPS:
            if not old.is_file(): continue
            with zipfile.ZipFile(old) as src:
                for filename in src.namelist():
                    if filename.endswith('.npy') and ('oof_' in filename or 'test_' in filename):
                        bn=Path(filename).name
                        if bn not in names:
                            dst.writestr(bn,src.read(filename))
                            names.add(bn)
            print('Included previous predictions from:',old)
            break
        for artifact in sorted(OUT_DIR.glob('*.npy')):
            if artifact.name not in names:
                dst.write(artifact,arcname=artifact.name)
                names.add(artifact.name)
        dst.write(MANIFEST,arcname=MANIFEST.name)
        exp=WORK_PROJECT/'experiments_overnight.csv'
        if exp.exists():dst.write(exp,arcname=exp.name)
    print('Updated final ZIP:',OUT_ZIP,'files:',len(names),'size(MB)',round(OUT_ZIP.stat().st_size/1e6,2))

def progress_paths(name):
    return dict(oof=PROGRESS_DIR/(name+'_partial_oof.npy'),
                tsum=PROGRESS_DIR/(name+'_partial_testsum.npy'),
                meta=PROGRESS_DIR/(name+'.json'))

def open_progress(name,config):
    paths=progress_paths(name)
    if paths['meta'].exists():
        old=json.loads(paths['meta'].read_text())
        assert old['config']==config, 'Config changed but member name reused!'
        arr=np.load(paths['oof']);tsum=np.load(paths['tsum'])
        done=list(map(int,old['done']))
        assert arr.shape==(len(train),) and tsum.shape==(len(test),)
        return arr,tsum,done,old.get('fold_details',[])
    return np.full(len(train),np.nan,dtype=np.float64),np.zeros(len(test),dtype=np.float64),[],[]

def save_progress(name,config,oof,tsum,done,details):
    paths=progress_paths(name)
    np.save(paths['oof'],oof)
    np.save(paths['tsum'],tsum)
    paths['meta'].write_text(json.dumps({'config':config,'done':sorted(done),'fold_details':details},indent=2))
    RUN_META['models'][name]={'complete':False,'folds_done':sorted(done),'fold_details':details}
    package_results() # keep a ZIP and manifest after each fold (partial OOF NOT published).

def official_save(name,oof,testp):
    assert oof.shape==(699635,) and testp.shape==(299844,)
    assert np.isfinite(oof).all() and np.isfinite(testp).all()
    assert np.all((oof>=0)&(oof<=1)) and np.all((testp>=0)&(testp<=1))
    np.save(OUT_DIR/f'oof_{name}.npy',oof.astype(np.float64))
    np.save(OUT_DIR/f'test_{name}.npy',testp.astype(np.float64))
    if (WORK_SRC/'utils.py').is_file():
        import utils as u
        # Team function checks shapes/AUC and stores to team output folder.
        if callable(getattr(u,'save_preds',None)):
            cwd=os.getcwd()
            try:
                os.chdir(WORK_SRC)
                u.save_preds(name,oof,testp,Y)
            finally:
                os.chdir(cwd)

def experiment_row(name,auc,notes):
    file=WORK_PROJECT/'experiments_overnight.csv'
    header=['date','person','name','cv_auc','lb_auc','notes']
    rows=[]
    if file.exists():
        with file.open(newline='',encoding='utf-8') as f:
            rows=list(csv.reader(f))[1:]
    rows=[r for r in rows if len(r)<3 or r[2]!=name]
    rows.append([datetime.now(timezone.utc).strftime('%Y-%m-%d'),'quan',name,f'{auc:.9f}','',notes])
    with file.open('w',newline='',encoding='utf-8') as f:
        w=csv.writer(f);w.writerow(header);w.writerows(rows)

def fold_summary(oof):
    return [float(roc_auc_score(Y[FOLDS==i],oof[FOLDS==i])) for i in range(5)]

# Predict with GPU array where CuPy is installed; fallback to numpy (warning is benign).
try:
    import cupy as cp
    HAS_CUPY=True
except ImportError:
    HAS_CUPY=False

def predict_proba(model,X):
    if HAS_CUPY:
        try:
            gpu=cp.asarray(X)
            prob=model.predict_proba(gpu)[:,1]
            ret=cp.asnumpy(prob) if isinstance(prob,cp.ndarray) else np.asarray(prob)
            del gpu,prob
            return ret
        except (RuntimeError,ValueError,TypeError) as exc:
            print('GPU prediction fallback (training stays on GPU):',str(exc)[:250])
    # Warning about mismatched devices is performance-only, not a correctness issue.
    return model.predict_proba(X)[:,1]

print('Checkpoint directory:',PROGRESS_DIR)
print('CuPy:',HAS_CUPY)

Checkpoint directory: /kaggle/working/cs116-kaggle-s6e10/progress_overnight
CuPy: True


In [5]:
# ===== MỘT LƯỢT QUA ĐÊM: 2 THÍ NGHIỆM ĐỘC LẬP =====
# Save & Run All executes both experiments in this one cell, sequentially on GPU 0.

def train_member(name, mode):
    begin=time.perf_counter()
    is_context=(mode=='context')
    te_frame=TE_CONTEXT if is_context else TE_BASE
    config='xgb-hist-gpu0-20261016-lr0.035-d6-child10-sub85-col85-lambda5-teauto-cv5-es10%-120-max3000-'+mode
    oof,tsum,done,details=open_progress(name,config)
    print('\n'+'='*75+'\nSTART',name,'| already completed folds:',done,flush=True)
    for k in range(5):
        if k in done: continue
        t0=time.perf_counter()
        tr=np.flatnonzero(FOLDS!=k)
        va=np.flatnonzero(FOLDS==k)
        fit,es=train_test_split(tr,test_size=0.10,stratify=Y[tr],random_state=42+k)

        # 1) Internal early-stopping: label-based features trained only on fit set.
        en=encoder(42+k)
        te_fit=en.fit_transform(te_frame.iloc[fit],Y[fit]).astype(np.float32)
        te_es=en.transform(te_frame.iloc[es]).astype(np.float32)
        if is_context:
            X_fit=np.concatenate([BASE_ARRAY[fit],te_fit,EXTRA_ARRAY[fit]],axis=1)
            X_es=np.concatenate([BASE_ARRAY[es],te_es,EXTRA_ARRAY[es]],axis=1)
        else:
            X_fit=np.concatenate([BASE_ARRAY[fit],te_fit],axis=1)
            X_es=np.concatenate([BASE_ARRAY[es],te_es],axis=1)
        del en,te_fit,te_es
        selector=learner(MAX_TREES,42+k,True)
        selector.fit(X_fit,Y[fit],eval_set=[(X_es,Y[es])],verbose=False)
        best=int(selector.best_iteration)+1
        print(f'  Fold {k}: internal best_trees={best}',flush=True)
        del selector,X_fit,X_es
        gc.collect()

        # 2) Outer train: encoder cross-fits on the permitted 4 folds only.
        en=encoder(42+k)
        te_tr=en.fit_transform(te_frame.iloc[tr],Y[tr]).astype(np.float32)
        te_va=en.transform(te_frame.iloc[va]).astype(np.float32)
        te_test=en.transform(te_frame.iloc[n_train:]).astype(np.float32)
        if is_context:
            X_tr=np.concatenate([BASE_ARRAY[tr],te_tr,EXTRA_ARRAY[tr]],axis=1)
            X_va=np.concatenate([BASE_ARRAY[va],te_va,EXTRA_ARRAY[va]],axis=1)
            X_test=np.concatenate([BASE_ARRAY[n_train:],te_test,EXTRA_ARRAY[n_train:]],axis=1)
        else:
            X_tr=np.concatenate([BASE_ARRAY[tr],te_tr],axis=1)
            X_va=np.concatenate([BASE_ARRAY[va],te_va],axis=1)
            X_test=np.concatenate([BASE_ARRAY[n_train:],te_test],axis=1)
        del en,te_tr,te_va,te_test
        gc.collect()

        mdl=learner(best,42+k,False)
        mdl.fit(X_tr,Y[tr],verbose=False)
        pred_va=np.asarray(predict_proba(mdl,X_va),dtype=np.float64)
        pred_test=np.asarray(predict_proba(mdl,X_test),dtype=np.float64)
        assert pred_va.shape==(len(va),) and pred_test.shape==(len(test),)
        assert np.isfinite(pred_va).all() and np.isfinite(pred_test).all()
        oof[va]=pred_va
        tsum+=pred_test
        auc=float(roc_auc_score(Y[va],pred_va))
        details.append({'fold':k,'auc':auc,'best_trees':best,'minutes':round((time.perf_counter()-t0)/60,2)})
        done.append(k)
        # Save checkpoint after EVERY fold before proceeding.
        save_progress(name,config,oof,tsum,done,details)
        print(f'  Fold {k} AUC={auc:.6f}, minutes={details[-1]["minutes"]} — checkpoint OK',flush=True)
        del mdl,X_tr,X_va,X_test,pred_va,pred_test
        gc.collect()
    assert sorted(done)==[0,1,2,3,4]
    avg_test=tsum/5
    official_save(name,oof,avg_test)
    auc=float(roc_auc_score(Y,oof))
    RUN_META['models'][name]={'complete':True,'overall_oof_auc':auc,
                             'fold_aucs':fold_summary(oof),'fold_details':details,
                             'minutes':round((time.perf_counter()-begin)/60,2)}
    experiment_row(name,auc,'GPU XGB; exact shared folds; nested TargetEncoder; train-only early stopping; '+mode)
    package_results()
    print('\nCOMPLETE',name,'OOF AUC:',round(auc,9),'fold:',fold_summary(oof),flush=True)
    return oof

summary={}
if RUN_XGB_TE:
    summary['quan_xgb_te_v1']=train_member('quan_xgb_te_v1','base')
if RUN_XGB_CONTEXT:
    summary['quan_xgb_context_v1']=train_member('quan_xgb_context_v1','context')

print('\n'+'='*60)
print('OVERNIGHT FINISHED:',OUT_ZIP)
for name,oof in summary.items():
    print(name, 'overall AUC',float(roc_auc_score(Y,oof)))
if len(summary)==2:
    keys=list(summary)
    print('OOF pair correlation:',np.corrcoef(summary[keys[0]],summary[keys[1]])[0,1])
print('Use separately as team members — DO NOT self-stack or submit.')
with zipfile.ZipFile(OUT_ZIP) as final_zip:
    for name in summary:
        expected=[f'oof_{name}.npy', f'test_{name}.npy']
        for fn in expected:
            assert fn in final_zip.namelist(), f'MISSING: {fn}'
    print('VERIFIED: all completed member files are inside the ZIP.')


START quan_xgb_te_v1 | already completed folds: []
  Fold 0: internal best_trees=1496
Included previous predictions from: /kaggle/input/notebooks/quannh206/cs116/S6E10_GPU_members.zip
Updated final ZIP: /kaggle/working/S6E10_Overnight_members.zip files: 6 size(MB) 19.69
  Fold 0 AUC=0.960005, minutes=0.7 — checkpoint OK
  Fold 1: internal best_trees=880
Included previous predictions from: /kaggle/input/notebooks/quannh206/cs116/S6E10_GPU_members.zip
Updated final ZIP: /kaggle/working/S6E10_Overnight_members.zip files: 6 size(MB) 19.69
  Fold 1 AUC=0.958583, minutes=0.55 — checkpoint OK
  Fold 2: internal best_trees=772
Included previous predictions from: /kaggle/input/notebooks/quannh206/cs116/S6E10_GPU_members.zip
Updated final ZIP: /kaggle/working/S6E10_Overnight_members.zip files: 6 size(MB) 19.69
  Fold 2 AUC=0.960537, minutes=0.53 — checkpoint OK
  Fold 3: internal best_trees=850
Included previous predictions from: /kaggle/input/notebooks/quannh206/cs116/S6E10_GPU_members.zip
Upd

### Đọc kết quả sáng mai
- **Versions → Output Files → `S6E10_Overnight_members.zip`** chứa **2 file `.npy`/member** + `S6E10_Overnight_manifest.json` và `experiments_overnight.csv`.
- `quan_xgb_te_v1` và `quan_xgb_context_v1` là **2 member riêng**, không blend.
- Nếu output ZIP có predictions cũ, các file cũ chỉ được **copy nguyên vẹn**, không retrain.
- Cần kiểm tra `"complete": true` cho cả 2 member trong manifest, OOF đầy đủ 5 fold trước khi gửi cho nhóm.
- Chú ý: XGBoost + TE base Fold 0 trong notebook này dùng seed `42 + k`; nếu khác pilot do khác random seed ở outer fold hoặc XGBoost, không coi là lỗi. Đánh giá theo **Overall 5-Fold OOF AUC**.
- AUC tăng ở member không đảm bảo stack tăng: người phụ trách stacking cần thử đóng góp riêng của từng member.